# Resize + Padding


## 0. Import Library & Path


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

INPUT_ROOT  = '/content/drive/MyDrive/Tugas Akhir/FINAL_WN_val_test'         # hasil Tahap 1
OUTPUT_ROOT = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_WN_val_test'


## 1. Konfigurasi


In [ ]:
TARGET_SIZE   = (224, 224)
PADDING_COLOR = (0, 0, 0)   # hitam, untuk citra RGB
MASK_PADDING  = 0           # untuk mask grayscale


## 2. Fungsi Resize + Padding


In [ ]:
def resize_with_padding(
    img: Image.Image,
    target_size: tuple = (224, 224),
    padding_color=(0, 0, 0),
    resample=Image.LANCZOS,
) -> Image.Image:
    target_w, target_h = target_size
    orig_w, orig_h = img.size

    # skala terkecil agar aspect ratio terjaga
    scale = min(target_w / orig_w, target_h / orig_h)
    new_w, new_h = int(orig_w * scale), int(orig_h * scale)
    img_resized = img.resize((new_w, new_h), resample)

    # canvas hitam target_size, citra ditempel di tengah
    canvas = Image.new(img.mode, (target_w, target_h), color=padding_color)
    paste_x = (target_w - new_w) // 2
    paste_y = (target_h - new_h) // 2
    canvas.paste(img_resized, (paste_x, paste_y))

    return canvas


## 3. Visualisasi Contoh (Before vs After)


In [ ]:
def visualize_padding_samples(input_root: str, n_samples: int = 4) -> None:
    candidates = [p for p in Path(input_root).rglob('*.png') if 'masks' not in p.parts]

    if not candidates:
        print('Tidak ada citra ditemukan di INPUT_ROOT.')
        return

    samples = candidates[:n_samples]
    fig, axes = plt.subplots(2, len(samples), figsize=(4 * len(samples), 8))

    for col, path in enumerate(samples):
        img_before = Image.open(path)
        img_after  = resize_with_padding(img_before, TARGET_SIZE, PADDING_COLOR)

        axes[0, col].imshow(img_before)
        axes[0, col].set_title(f'Before\nSize: {img_before.size}', fontsize=9)
        axes[0, col].axis('off')

        axes[1, col].imshow(img_after)
        axes[1, col].set_title(f'After\nSize: {img_after.size}', fontsize=9)
        axes[1, col].axis('off')

    plt.tight_layout()
    plt.show()

visualize_padding_samples(INPUT_ROOT)


## 4. Batch Processing — Seluruh Dataset


In [ ]:
def run_padding_pipeline(input_root: str, output_root: str, target_size: tuple) -> dict:
    image_files = list(Path(input_root).rglob('*.png'))
    summary = {'success': 0, 'fail': 0, 'total': len(image_files)}

    for img_file in tqdm(image_files, desc='Resize+Padding', unit='img'):
        try:
            # file di folder masks -> NEAREST + padding 0 (grayscale)
            is_mask = 'masks' in img_file.relative_to(input_root).parts
            img     = Image.open(img_file)

            if is_mask:
                img_padded = resize_with_padding(img, target_size, MASK_PADDING, Image.NEAREST)
            else:
                img_padded = resize_with_padding(img, target_size, PADDING_COLOR, Image.LANCZOS)

            dst_path = Path(output_root) / img_file.relative_to(input_root)
            dst_path.parent.mkdir(parents=True, exist_ok=True)
            img_padded.save(dst_path, format='PNG')
            summary['success'] += 1
        except Exception as e:
            summary['fail'] += 1
            print(f'Gagal: {img_file.name} -> {e}')

    return summary

summary_padding = run_padding_pipeline(INPUT_ROOT, OUTPUT_ROOT, TARGET_SIZE)


In [ ]:
print(f"{summary_padding['success']}/{summary_padding['total']} citra berhasil di-resize+padding.")
print(f'Output: {OUTPUT_ROOT}')
